### **Student Information**
GitHub ID:Jerrykuo0701

Kaggle name:KuoYaowei

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

**Syntax:** `#` creates the largest heading (H1).

---
**Syntax:** `---` creates a horizontal rule (a separator line).

## 1. Model Development (10 pts Required)

**Syntax:** `##` creates a secondary heading (H2).

**Describe briefly each section, you can add graphs/charts to support your explanations.**

### 1.1 Preprocessing Steps

**Data loading and merging**
- The three files were merged on `post_id`: `final_posts.json` (text), `data_identification.csv` (train/test split) and `emotion.csv` (labels).
- This gives 3,360 training and 1,440 test posts, with no missing or empty texts.

**Data exploration**
- Six emotions with an imbalanced distribution: joy 26%, anger 20%, sadness 19%, disgust 13.5%, surprise 13% and fear only 8.6%. Since the metric is macro F1, the rare classes matter as much as the common ones.
- The texts come from mixed sources: Reddit comments (names replaced by `[NAME]`), news-style sentences and a few non-English lines. The median length is 13 words.

**Text cleaning (different for each model type)**
- TF-IDF baseline: lowercasing, removing URLs, replacing `@mentions` with `@user`, and collapsing whitespace.
- Transformers (RoBERTa, DeBERTa): only URLs and extra whitespace were removed. Casing and punctuation were kept on purpose, because they carry emotion (for example "WTF!" or "SHOCKED!"), and the pre-trained tokenizers already handle them.

**Submission format**
- `sampleSubmission.csv` uses the column `post_id`, but Kaggle's scorer expects `id`. My first submission failed with "ID column id not found", so the column was renamed to `id`.

### 1.2 Feature Engineering Steps

**Baseline: TF-IDF features**
- Word 1-2 grams: capture single words and short phrases such as "not happy".
- Character 2-5 grams (`char_wb`): handle misspellings, slang and stretched words (for example "sooo" or "angrrry"), which are common in social-media text.
- Both use `min_df=2` (drop features seen in only one post) and `sublinear_tf=True` (dampen repeated words), and are concatenated with `make_union`.
- With a Linear SVM this reached only 0.4993 CV macro F1 and 0.4873 on the public leaderboard, below the 0.5 baseline. TF-IDF only counts which words appear, so it misses tone and meaning, which this task depends on.

**Final models: pre-trained Transformer representations**
- Instead of hand-made features, each post is converted into sub-word tokens by the tokenizer of the pre-trained model (`roberta-base` and `microsoft/deberta-v3-base`).
- Texts are truncated to 128 tokens; since the median post is only 13 words, this covers almost every post in full.
- The pre-trained model then produces a contextual representation of the whole sentence, so the same word can mean different things depending on its context (for example "offensive" in a sports tweet versus an insult).
- **Class weights** (square root of inverse class frequency) are applied in the loss function, so that rare emotions such as fear (8.6%) are not ignored by the model.

### 1.3 Explanation of Your Model

### 1.3 Explanation of Your Model

**Development process**

| Model | CV macro F1 | Public LB |
| --- | ---: | ---: |
| TF-IDF + Linear SVM (baseline) | 0.4993 | 0.4873 |
| RoBERTa-base | 0.5957 | 0.5768 |
| RoBERTa + DeBERTa ensemble | 0.6048 | 0.5962 |
| RoBERTa + 2x DeBERTa ensemble | 0.6159 | **0.6034** |
| All 5 models ensemble | **0.6196** | 0.6020 |

**1. Baseline: TF-IDF + Linear SVM**
- A Linear SVM with `class_weight="balanced"` on the word and character TF-IDF features.
- It scored 0.4873 on the public leaderboard, below the 0.5 baseline, so a stronger model was needed.

**2. Fine-tuned Transformers (trained on a Kaggle T4 GPU)**
- Pre-trained `roberta-base` and `microsoft/deberta-v3-base` with a new 6-class classification head.
- **5-fold stratified cross-validation:** each fold trains on 80% of the training data and is evaluated on the other 20%; the test predictions of the 5 folds are averaged.
- **Training setup:** AdamW, linear learning-rate schedule with 10% warmup, gradient clipping, mixed precision (FP16), batch size 16, and the class-weighted loss from 1.2.
- **Best epoch per fold:** after every epoch the model is scored on its validation fold with macro F1, and the best epoch is kept.
- **Tuning DeBERTa:** with 4 epochs and learning rate 2e-5 it was still improving at the last epoch (OOF 0.5819). With 6 epochs and 3e-5 it rose to 0.6006. The same change for RoBERTa gave only 0.5957 → 0.5971, because RoBERTa had already peaked by epochs 2-4.

**3. Ensemble (final model)**
- The class probabilities of several models are averaged. Different models make different mistakes, so the average corrects some of them: every ensemble scored higher than its best single model.
- I trained five models in total: RoBERTa (4 and 6 epochs) and DeBERTa (4 epochs, 6 epochs, and 6 epochs with another random seed).

**4. Final submissions and their limits**
- Changing only the random seed moved DeBERTa from 0.6006 to 0.6045, so differences smaller than about 0.004 are noise, not real improvements.
- Choosing the best epoch, and the best of 26 model combinations, both make the CV score slightly optimistic. The best combination (0.6203) was therefore **not** used; it was only 0.0007 above the average of all five models.
- I selected two final submissions: the 3-model ensemble (best public score, 0.6034) and the all-5-model ensemble (best CV, chosen in advance without selection, 0.6196).
- I deliberately didn't use models already fine-tuned on emotion datasets such as GoEmotions, because part of this data looks like it comes from similar Reddit sources, which could leak test labels.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI
- While working on this assignment, I encountered some parts where the code structure was unclear or contained errors, so I sought advice from an AI. 
---

| Model | CV macro F1 | Public LB | Private LB |
| --- | ---: | ---: | ---: |
| TF-IDF + Linear SVM (baseline) | 0.4993 | 0.4873 | 0.4905 |
| RoBERTa-base | 0.5957 | 0.5768 | 0.6455 |
| RoBERTa + DeBERTa ensemble | 0.6048 | 0.5962 | 0.6343 |
| RoBERTa + 2x DeBERTa ensemble | 0.6159 | 0.6034 | 0.6359 |
| All 5 models ensemble (final) | 0.6196 | 0.6020 | **0.6376** |

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [2]:
# Preprocessing Steps: Make it to load the three files, merge them by post_id, and clean the text
import json, re
import numpy as np
import pandas as pd

import os
import glob
# Data path: search Kaggle's input folder for the competition files; fall back to the local folder outside the repo
found = glob.glob("/kaggle/input/**/final_posts.json", recursive=True)
DATA_DIR = os.path.dirname(found[0]) if found else "../2026-dm-lab-2-private-competition"
OUT_DIR = "/kaggle/working" if found else DATA_DIR   # where submissions are written
print("DATA_DIR:", DATA_DIR)
SEED = 42

posts = pd.DataFrame(json.load(open(f"{DATA_DIR}/final_posts.json", encoding="utf-8")))
split = pd.read_csv(f"{DATA_DIR}/data_identification.csv")
labels = pd.read_csv(f"{DATA_DIR}/emotion.csv")

# Merge text, train/test membership and labels (test rows have no label)
df = posts.merge(split, on="post_id").merge(labels, on="post_id", how="left")

def clean_text(t):
    t = t.lower()                                  # lowercase
    t = re.sub(r"https?://\S+", " ", t)            # drop URLs
    t = re.sub(r"@\w+", " @user ", t)              # replace user mentions
    return re.sub(r"\s+", " ", t).strip()          # collapse whitespace

df["clean"] = df["text"].map(clean_text)
train_df = df[df["set"] == "train"].reset_index(drop=True)
test_df = df[df["set"] == "test"].reset_index(drop=True)
print(train_df.shape, test_df.shape)
print(train_df["label"].value_counts())

DATA_DIR: /kaggle/input/competitions/2026-dm-lab-2-private-competition
(3360, 5) (1440, 5)
label
joy         875
anger       674
sadness     628
disgust     454
surprise    441
fear        288
Name: count, dtype: int64


## 2. Feature Engineering Steps

In [3]:
# Feature engineering: word 1-2 grams plus character 2-5 grams (character n-grams help with misspellings, slang and the mixed sources)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_union

features = make_union(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
)

## 3. Model Implementation Steps

In [4]:
# Model: Linear SVM with balanced class weights (fear is only 8.6% of training data)
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

model = make_pipeline(features, LinearSVC(C=0.5, class_weight="balanced", random_state=SEED))

# 5-fold cross-validation on the training set, scored with macro F1 like Kaggle
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores = cross_val_score(model, train_df["clean"], train_df["label"], cv=cv, scoring="f1_macro")
print(f"CV macro F1: {scores.mean():.4f} +/- {scores.std():.4f}")

# Fit on all training data and write the submission (columns: id, label)
model.fit(train_df["clean"], train_df["label"])
# Kaggle's solution file expects the column "id" (as on the Overview page), not "post_id"
submission = pd.DataFrame({"id": test_df["post_id"], "label": model.predict(test_df["clean"])})
submission.to_csv(f"{OUT_DIR}/submission_tfidf_svm.csv", index=False)
print(submission.shape, submission["label"].value_counts().to_dict())

CV macro F1: 0.4993 +/- 0.0172
(1440, 2) {'joy': 395, 'anger': 325, 'sadness': 269, 'surprise': 178, 'disgust': 165, 'fear': 108}


### Final model: fine-tuned RoBERTa (run on Kaggle with a T4 GPU)
The TF-IDF baseline above scored 0.4873 on the public leaderboard, below the 0.5 baseline, so the final model fine-tunes `roberta-base` with 5-fold cross-validation.

In [5]:
# Fine-tune roberta-base with 5-fold CV (needs a GPU: run on Kaggle)
# Each fold keeps its best epoch by validation macro F1; test predictions are averaged over folds.
import random, gc
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)
from sklearn.metrics import f1_score

MODEL_NAME = "roberta-base"          # general pre-trained model (MIT license)
N_FOLDS, EPOCHS, LR, MAX_LEN, BATCH = 5, 4, 2e-5, 128, 16
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def set_seed(seed):
    # Fix every random source so the run can be reproduced
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# Light cleaning only: RoBERTa uses casing and punctuation, so the original text is kept
def light_clean(s):
    return s.str.replace(r"https?://\S+", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()

train_text, test_text = light_clean(train_df["text"]), light_clean(test_df["text"])
LABELS = sorted(train_df["label"].unique())
label2id = {l: i for i, l in enumerate(LABELS)}
y = train_df["label"].map(label2id).to_numpy()

# Features: RoBERTa sub-word tokens
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = labels
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        item = {k: self.enc[k][i] for k in self.enc}
        if self.labels is not None:
            item["labels"] = int(self.labels[i])
        return item

# Mild class weights (square root of inverse frequency) to help rare classes such as fear
counts = np.bincount(y, minlength=len(LABELS))
class_weights = torch.tensor(np.sqrt(len(y) / (len(LABELS) * counts)), dtype=torch.float, device=device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

@torch.no_grad()
def predict_proba(model, loader):
    # Softmax probabilities for every row of a loader
    model.eval(); out = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
            logits = model(**batch).logits
        out.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(out)

oof = np.zeros((len(train_df), len(LABELS)))
test_proba = np.zeros((len(test_df), len(LABELS)))
test_loader = DataLoader(TextDataset(test_text), batch_size=64, collate_fn=collator)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_text, y)):
    set_seed(SEED + fold)
    tr_loader = DataLoader(TextDataset(train_text.iloc[tr_idx], y[tr_idx]),
                           batch_size=BATCH, shuffle=True, collate_fn=collator)
    va_loader = DataLoader(TextDataset(train_text.iloc[va_idx]), batch_size=64, collate_fn=collator)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(LABELS)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = EPOCHS * len(tr_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=device == "cuda")

    best_f1 = -1.0
    for epoch in range(EPOCHS):
        model.train()
        for batch in tr_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                loss = loss_fn(model(**batch).logits.float(), labels)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()

        va_proba = predict_proba(model, va_loader)
        f1 = f1_score(y[va_idx], va_proba.argmax(1), average="macro")
        print(f"fold {fold} epoch {epoch + 1}: val macro F1 = {f1:.4f}")
        if f1 > best_f1:                       # keep this fold's best epoch
            best_f1 = f1
            oof[va_idx] = va_proba
            fold_test = predict_proba(model, test_loader)
    test_proba += fold_test / N_FOLDS
    print(f"fold {fold} best: {best_f1:.4f}")
    del model, optimizer; gc.collect(); torch.cuda.empty_cache()

print(f"\nOOF macro F1 (5-fold CV): {f1_score(y, oof.argmax(1), average='macro'):.4f}")

# Submission: Kaggle expects the columns id, label
submission = pd.DataFrame({"id": test_df["post_id"], "label": [LABELS[i] for i in test_proba.argmax(1)]})
submission.to_csv(f"{OUT_DIR}/submission_roberta.csv", index=False)
print(submission.shape, submission["label"].value_counts().to_dict())

device: cuda


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (13 outputs left out)

fold 3 epoch 1: val macro F1 = 0.5814
fold 3 epoch 2: val macro F1 = 0.6051
fold 3 epoch 3: val macro F1 = 0.6186
fold 3 epoch 4: val macro F1 = 0.6038
fold 3 best: 0.6186


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_50/3298749096.py:76: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(

fold 4 epoch 1: val macro F1 = 0.4657
fold 4 epoch 2: val macro F1 = 0.5657
fold 4 epoch 3: val macro F1 = 0.5800
fold 4 epoch 4: val macro F1 = 0.5751
fold 4 best: 0.5800

OOF macro F1 (5-fold CV): 0.5957
(1440, 2) {'joy': 366, 'anger': 294, 'sadness': 280, 'surprise': 196, 'disgust': 165, 'fear': 139}


In [6]:
# Keep the RoBERTa predictions from 3.2 before training the next model
roberta_oof, roberta_test = oof.copy(), test_proba.copy()
print("RoBERTa OOF macro F1:", round(f1_score(y, roberta_oof.argmax(1), average="macro"), 4))

RoBERTa OOF macro F1: 0.5957


In [8]:
# Same 5-fold fine-tuning, wrapped in a function so any pre-trained model can be plugged in
def run_cv(model_name, epochs=4, lr=2e-5, use_amp=True):
    amp = use_amp and device == "cuda"          # mixed precision only on GPU
    tok = AutoTokenizer.from_pretrained(model_name)
    coll = DataCollatorWithPadding(tok)
 
    class DS(Dataset):
        # Tokenized texts (and labels for training rows)
        def __init__(self, texts, labels=None):
            self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN)
            self.labels = labels
        def __len__(self):
            return len(self.enc["input_ids"])
        def __getitem__(self, i):
            item = {k: self.enc[k][i] for k in self.enc}
            if self.labels is not None:
                item["labels"] = int(self.labels[i])
            return item
 
    @torch.no_grad()
    def proba(model, loader):
        # Softmax probabilities for every row of a loader
        model.eval(); out = []
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                logits = model(**batch).logits
            out.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
        return np.concatenate(out)
 
    oof_p = np.zeros((len(train_df), len(LABELS)))
    test_p = np.zeros((len(test_df), len(LABELS)))
    te_loader = DataLoader(DS(test_text), batch_size=64, collate_fn=coll)
    folds = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)   # same folds as 3.2
 
    for fold, (tr_idx, va_idx) in enumerate(folds.split(train_text, y)):
        set_seed(SEED + fold)
        tr_loader = DataLoader(DS(train_text.iloc[tr_idx], y[tr_idx]), batch_size=BATCH,
                               shuffle=True, collate_fn=coll)
        va_loader = DataLoader(DS(train_text.iloc[va_idx]), batch_size=64, collate_fn=coll)
 
        # .float(): some checkpoints (e.g. DeBERTa-v3) are stored in FP16; mixed-precision training needs FP32 weights
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(LABELS)).float().to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = epochs * len(tr_loader)
        scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        scaler = torch.amp.GradScaler("cuda", enabled=amp)
 
        best_f1 = -1.0
        for epoch in range(epochs):
            model.train()
            for batch in tr_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                labels = batch.pop("labels")
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                    loss = loss_fn(model(**batch).logits.float(), labels)   # same class weights as 3.2
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update(); scheduler.step()
 
            va_p = proba(model, va_loader)
            f1 = f1_score(y[va_idx], va_p.argmax(1), average="macro")
            print(f"{model_name} fold {fold} epoch {epoch + 1}: val macro F1 = {f1:.4f}")
            if f1 > best_f1:                     # keep this fold's best epoch
                best_f1 = f1
                oof_p[va_idx] = va_p
                fold_test = proba(model, te_loader)
        test_p += fold_test / N_FOLDS
        del model, optimizer; gc.collect(); torch.cuda.empty_cache()
 
    print(f"{model_name} OOF macro F1: {f1_score(y, oof_p.argmax(1), average='macro'):.4f}")
    return oof_p, test_p
 
deberta_oof, deberta_test = run_cv("microsoft/deberta-v3-base")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

microsoft/deberta-v3-base fold 0 epoch 1: val macro F1 = 0.2764
microsoft/deberta-v3-base fold 0 epoch 2: val macro F1 = 0.4874
microsoft/deberta-v3-base fold 0 epoch 3: val macro F1 = 0.5521
microsoft/deberta-v3-base fold 0 epoch 4: val macro F1 = 0.5811


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

microsoft/deberta-v3-base fold 1 epoch 1: val macro F1 = 0.2546
microsoft/deberta-v3-base fold 1 epoch 2: val macro F1 = 0.5264
microsoft/deberta-v3-base fold 1 epoch 3: val macro F1 = 0.5617
microsoft/deberta-v3-base fold 1 epoch 4: val macro F1 = 0.5543


... (5 outputs left out)

microsoft/deberta-v3-base fold 3 epoch 1: val macro F1 = 0.3712
microsoft/deberta-v3-base fold 3 epoch 2: val macro F1 = 0.5641
microsoft/deberta-v3-base fold 3 epoch 3: val macro F1 = 0.5848
microsoft/deberta-v3-base fold 3 epoch 4: val macro F1 = 0.5877


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

microsoft/deberta-v3-base fold 4 epoch 1: val macro F1 = 0.3498
microsoft/deberta-v3-base fold 4 epoch 2: val macro F1 = 0.5621
microsoft/deberta-v3-base fold 4 epoch 3: val macro F1 = 0.5510
microsoft/deberta-v3-base fold 4 epoch 4: val macro F1 = 0.5729
microsoft/deberta-v3-base OOF macro F1: 0.5819


In [9]:
#Ensemble: average the class probabilities of RoBERTa and DeBERTa
ens_oof = (roberta_oof + deberta_oof) / 2
ens_test = (roberta_test + deberta_test) / 2
for name, p in [("RoBERTa", roberta_oof), ("DeBERTa", deberta_oof), ("Ensemble", ens_oof)]:
    print(f"{name:9s} OOF macro F1: {f1_score(y, p.argmax(1), average='macro'):.4f}")
 
# Submission: Kaggle expects the columns id, label
submission = pd.DataFrame({"id": test_df["post_id"], "label": [LABELS[i] for i in ens_test.argmax(1)]})
submission.to_csv(f"{OUT_DIR}/submission_ensemble.csv", index=False)
print(submission.shape, submission["label"].value_counts().to_dict())

RoBERTa   OOF macro F1: 0.5957
DeBERTa   OOF macro F1: 0.5819
Ensemble  OOF macro F1: 0.6048
(1440, 2) {'joy': 361, 'anger': 317, 'sadness': 276, 'surprise': 214, 'disgust': 153, 'fear': 119}


In [10]:
# DeBERTa trained longer: it was still improving at epoch 4, so this run uses 6 epochs and a slightly higher learning rate
deberta2_oof, deberta2_test = run_cv("microsoft/deberta-v3-base", epochs=6, lr=3e-5)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

microsoft/deberta-v3-base fold 0 epoch 1: val macro F1 = 0.3936
microsoft/deberta-v3-base fold 0 epoch 2: val macro F1 = 0.5750
microsoft/deberta-v3-base fold 0 epoch 3: val macro F1 = 0.5589
microsoft/deberta-v3-base fold 0 epoch 4: val macro F1 = 0.5859
microsoft/deberta-v3-base fold 0 epoch 5: val macro F1 = 0.5868
microsoft/deberta-v3-base fold 0 epoch 6: val macro F1 = 0.5841


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

microsoft/deberta-v3-base fold 1 epoch 1: val macro F1 = 0.3145
microsoft/deberta-v3-base fold 1 epoch 2: val macro F1 = 0.5626
microsoft/deberta-v3-base fold 1 epoch 3: val macro F1 = 0.5975
microsoft/deberta-v3-base fold 1 epoch 4: val macro F1 = 0.5735
microsoft/deberta-v3-base fold 1 epoch 5: val macro F1 = 0.5749
microsoft/deberta-v3-base fold 1 epoch 6: val macro F1 = 0.5723


... (5 outputs left out)

microsoft/deberta-v3-base fold 3 epoch 1: val macro F1 = 0.4284
microsoft/deberta-v3-base fold 3 epoch 2: val macro F1 = 0.5874
microsoft/deberta-v3-base fold 3 epoch 3: val macro F1 = 0.5910
microsoft/deberta-v3-base fold 3 epoch 4: val macro F1 = 0.5690
microsoft/deberta-v3-base fold 3 epoch 5: val macro F1 = 0.5796
microsoft/deberta-v3-base fold 3 epoch 6: val macro F1 = 0.5633


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

microsoft/deberta-v3-base fold 4 epoch 1: val macro F1 = 0.3694
microsoft/deberta-v3-base fold 4 epoch 2: val macro F1 = 0.5901
microsoft/deberta-v3-base fold 4 epoch 3: val macro F1 = 0.5770
microsoft/deberta-v3-base fold 4 epoch 4: val macro F1 = 0.5814
microsoft/deberta-v3-base fold 4 epoch 5: val macro F1 = 0.5857
microsoft/deberta-v3-base fold 4 epoch 6: val macro F1 = 0.5931
microsoft/deberta-v3-base OOF macro F1: 0.6006


In [11]:
# Compare ensembles on the out-of-fold predictions (equal-weight probability averages)
candidates = {
    "RoBERTa + DeBERTa (4 ep)":([roberta_oof, deberta_oof], [roberta_test, deberta_test]),
    "RoBERTa + DeBERTa (6 ep)":([roberta_oof, deberta2_oof], [roberta_test, deberta2_test]),
    "RoBERTa + both DeBERTa runs":([roberta_oof, deberta_oof, deberta2_oof],  [roberta_test, deberta_test, deberta2_test]),
}
scores = {}
for name, (oofs, tests) in candidates.items():
    scores[name] = f1_score(y, np.mean(oofs, axis=0).argmax(1), average="macro")
    print(f"{name:30s} OOF macro F1: {scores[name]:.4f}")

# Pick the ensemble with the best OOF score and write its submission
best = max(scores, key=scores.get)
best_test = np.mean(candidates[best][1], axis=0)
submission = pd.DataFrame({"id": test_df["post_id"], "label": [LABELS[i] for i in best_test.argmax(1)]})
submission.to_csv(f"{OUT_DIR}/submission_ensemble2.csv", index=False)
print("\nbest:", best, "->", submission.shape, submission["label"].value_counts().to_dict())

RoBERTa + DeBERTa (4 ep)       OOF macro F1: 0.6048
RoBERTa + DeBERTa (6 ep)       OOF macro F1: 0.6096
RoBERTa + both DeBERTa runs    OOF macro F1: 0.6159

best: RoBERTa + both DeBERTa runs -> (1440, 2) {'joy': 365, 'anger': 307, 'sadness': 281, 'surprise': 212, 'disgust': 164, 'fear': 111}


In [12]:
# RoBERTa with the settings that helped DeBERTa: 6 epochs and learning rate 3e-5
roberta2_oof, roberta2_test = run_cv("roberta-base", epochs=6, lr=3e-5)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


roberta-base fold 0 epoch 1: val macro F1 = 0.5517
roberta-base fold 0 epoch 2: val macro F1 = 0.5798
roberta-base fold 0 epoch 3: val macro F1 = 0.5693
roberta-base fold 0 epoch 4: val macro F1 = 0.5878
roberta-base fold 0 epoch 5: val macro F1 = 0.5797
roberta-base fold 0 epoch 6: val macro F1 = 0.5887


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


roberta-base fold 1 epoch 1: val macro F1 = 0.5089
roberta-base fold 1 epoch 2: val macro F1 = 0.5495
roberta-base fold 1 epoch 3: val macro F1 = 0.5537
roberta-base fold 1 epoch 4: val macro F1 = 0.5742
roberta-base fold 1 epoch 5: val macro F1 = 0.5612
roberta-base fold 1 epoch 6: val macro F1 = 0.5654


... (5 outputs left out)

roberta-base fold 3 epoch 1: val macro F1 = 0.5959
roberta-base fold 3 epoch 2: val macro F1 = 0.6009
roberta-base fold 3 epoch 3: val macro F1 = 0.6117
roberta-base fold 3 epoch 4: val macro F1 = 0.5914
roberta-base fold 3 epoch 5: val macro F1 = 0.6002
roberta-base fold 3 epoch 6: val macro F1 = 0.5908


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


roberta-base fold 4 epoch 1: val macro F1 = 0.5158
roberta-base fold 4 epoch 2: val macro F1 = 0.5669
roberta-base fold 4 epoch 3: val macro F1 = 0.5864
roberta-base fold 4 epoch 4: val macro F1 = 0.6013
roberta-base fold 4 epoch 5: val macro F1 = 0.5866
roberta-base fold 4 epoch 6: val macro F1 = 0.5839
roberta-base OOF macro F1: 0.5971


In [13]:
# DeBERTa (6 epochs, lr 3e-5) with a different random seed.
# The seed changes both the fold split and the weight initialisation, so this model makes different mistakes; every row is still predicted by a model that never trained on it.
SEED_BACKUP = SEED
SEED = 2026
deberta3_oof, deberta3_test = run_cv("microsoft/deberta-v3-base", epochs=6, lr=3e-5)
SEED = SEED_BACKUP          # restore the original seed

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

microsoft/deberta-v3-base fold 0 epoch 1: val macro F1 = 0.3814
microsoft/deberta-v3-base fold 0 epoch 2: val macro F1 = 0.6020
microsoft/deberta-v3-base fold 0 epoch 3: val macro F1 = 0.6117
microsoft/deberta-v3-base fold 0 epoch 4: val macro F1 = 0.5984
microsoft/deberta-v3-base fold 0 epoch 5: val macro F1 = 0.5895
microsoft/deberta-v3-base fold 0 epoch 6: val macro F1 = 0.5902


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

microsoft/deberta-v3-base fold 1 epoch 1: val macro F1 = 0.3047
microsoft/deberta-v3-base fold 1 epoch 2: val macro F1 = 0.5207
microsoft/deberta-v3-base fold 1 epoch 3: val macro F1 = 0.5586
microsoft/deberta-v3-base fold 1 epoch 4: val macro F1 = 0.5680
microsoft/deberta-v3-base fold 1 epoch 5: val macro F1 = 0.5749
microsoft/deberta-v3-base fold 1 epoch 6: val macro F1 = 0.5684


... (5 outputs left out)

microsoft/deberta-v3-base fold 3 epoch 1: val macro F1 = 0.2879
microsoft/deberta-v3-base fold 3 epoch 2: val macro F1 = 0.5646
microsoft/deberta-v3-base fold 3 epoch 3: val macro F1 = 0.5573
microsoft/deberta-v3-base fold 3 epoch 4: val macro F1 = 0.5648
microsoft/deberta-v3-base fold 3 epoch 5: val macro F1 = 0.5844
microsoft/deberta-v3-base fold 3 epoch 6: val macro F1 = 0.5899


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

microsoft/deberta-v3-base fold 4 epoch 1: val macro F1 = 0.2519
microsoft/deberta-v3-base fold 4 epoch 2: val macro F1 = 0.5416
microsoft/deberta-v3-base fold 4 epoch 3: val macro F1 = 0.6065
microsoft/deberta-v3-base fold 4 epoch 4: val macro F1 = 0.6186
microsoft/deberta-v3-base fold 4 epoch 5: val macro F1 = 0.6234
microsoft/deberta-v3-base fold 4 epoch 6: val macro F1 = 0.6143
microsoft/deberta-v3-base OOF macro F1: 0.6045


In [14]:
# Try every combination of at least two models (equal-weight probability average)
from itertools import combinations

models = {
    "RoBERTa (4ep)":(roberta_oof,  roberta_test),
    "RoBERTa (6ep)":(roberta2_oof, roberta2_test),
    "DeBERTa (4ep)":(deberta_oof,  deberta_test),
    "DeBERTa (6ep)":(deberta2_oof, deberta2_test),
    "DeBERTa (6ep, seed2)":(deberta3_oof, deberta3_test),
}
for name, (o, _) in models.items():
    print(f"{name:22s} single OOF macro F1: {f1_score(y, o.argmax(1), average='macro'):.4f}")

results = []
for r in range(2, len(models) + 1):
    for combo in combinations(models, r):
        avg_oof = np.mean([models[m][0] for m in combo], axis=0)
        results.append((f1_score(y, avg_oof.argmax(1), average="macro"), combo))
results.sort(reverse=True)
print("\nTop 5 ensembles by OOF macro F1:")
for score, combo in results[:5]:
    print(f"  {score:.4f}  {' + '.join(combo)}")

# Write the best ensemble as the submission
best_score, best_combo = results[0]
best_test = np.mean([models[m][1] for m in best_combo], axis=0)
submission = pd.DataFrame({"id": test_df["post_id"], "label": [LABELS[i] for i in best_test.argmax(1)]})
submission.to_csv(f"{OUT_DIR}/submission_ensemble3.csv", index=False)
print("\nbest:", best_score, best_combo, submission["label"].value_counts().to_dict())

RoBERTa (4ep)          single OOF macro F1: 0.5957
RoBERTa (6ep)          single OOF macro F1: 0.5971
DeBERTa (4ep)          single OOF macro F1: 0.5819
DeBERTa (6ep)          single OOF macro F1: 0.6006
DeBERTa (6ep, seed2)   single OOF macro F1: 0.6045

Top 5 ensembles by OOF macro F1:
  0.6203  RoBERTa (6ep) + DeBERTa (4ep) + DeBERTa (6ep, seed2)
  0.6196  RoBERTa (4ep) + RoBERTa (6ep) + DeBERTa (4ep) + DeBERTa (6ep) + DeBERTa (6ep, seed2)
  0.6195  RoBERTa (6ep) + DeBERTa (6ep) + DeBERTa (6ep, seed2)
  0.6188  RoBERTa (4ep) + RoBERTa (6ep) + DeBERTa (6ep) + DeBERTa (6ep, seed2)
  0.6178  RoBERTa (4ep) + DeBERTa (4ep) + DeBERTa (6ep) + DeBERTa (6ep, seed2)

best: 0.6203457715597207 ('RoBERTa (6ep)', 'DeBERTa (4ep)', 'DeBERTa (6ep, seed2)') {'joy': 375, 'anger': 304, 'sadness': 282, 'surprise': 207, 'disgust': 167, 'fear': 105}


In [15]:
# Final ensemble: all five models, equal weights. Chosen in advance rather than by picking the best of 26 combinations, so its OOF score (0.6196) is not inflated by the selection.
all_test = np.mean([t for _, t in models.values()], axis=0)
submission = pd.DataFrame({"id": test_df["post_id"], "label": [LABELS[i] for i in all_test.argmax(1)]})
submission.to_csv(f"{OUT_DIR}/submission_ensemble_all5.csv", index=False)
print(submission.shape, submission["label"].value_counts().to_dict())

(1440, 2) {'joy': 377, 'anger': 304, 'sadness': 280, 'surprise': 201, 'disgust': 173, 'fear': 105}
